In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import shutil
import random
import pandas as pd
from collections import Counter, defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

BASE_DIR = Path("/content/drive/MyDrive/DIS_PROJESI_007")

INTRAORAL_RAW = BASE_DIR / "data_extracted" / "intraoral_caries" / "Dataset"
INTRAORAL_PREPARED = BASE_DIR / "data_prepared" / "intraoral_caries_yolo"

RESULTS_DIR = BASE_DIR / "results" / "intraoral_caries_prepare"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Intraoral raw:", INTRAORAL_RAW)
print("Intraoral prepared:", INTRAORAL_PREPARED)
print("Raw var mı?:", INTRAORAL_RAW.exists())

Intraoral raw: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset
Intraoral prepared: /content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo
Raw var mı?: True


In [3]:
def print_tree(root_path, max_depth=4, max_files_per_folder=10):
    root_path = Path(root_path)

    print("\nKlasör yapısı:", root_path)
    print("=" * 100)

    for current_root, dirs, files in os.walk(root_path):
        current_root = Path(current_root)
        depth = len(current_root.relative_to(root_path).parts)

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth
        print(f"{indent}{current_root.name}/")

        file_indent = "    " * (depth + 1)
        for f in files[:max_files_per_folder]:
            print(f"{file_indent}{f}")

        if len(files) > max_files_per_folder:
            print(f"{file_indent}... {len(files) - max_files_per_folder} dosya daha")

print_tree(INTRAORAL_RAW, max_depth=4)


Klasör yapısı: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset
Dataset/
    Annotations/
        Darknet_YOLO/
            no_retractors/
                Frontal/
                    anonymous_003-007-1214-00_1732862917124_Frontal_View.txt
                    anonymous_003-007-673-00_1729239709286_Frontal_View.txt
                    anonymous_003-007-848-00_1730185308013_Frontal_View.txt
                    anonymous_003-007-870-00_1730264018655_Frontal_View.txt
                    anonymous_003-007-950-00_1730792603619_Frontal_View.txt
                    anonymous_003-008-1052-00_1731329608354_Frontal_View.txt
                    anonymous_003-008-1134-00_1732531184012_Frontal_View.txt
                    anonymous_003-008-1140-00_1732538036389_Frontal_View.txt
                    anonymous_003-008-1188-00_1732714947165_Frontal_View.txt
                    anonymous_003-008-563-00_1728904641646_Frontal_View.txt
                    ... 6 dosya daha
   

In [4]:
IMAGES_DIR = INTRAORAL_RAW / "Images"
YOLO_ANN_DIR = INTRAORAL_RAW / "Annotations" / "Darknet_YOLO"

image_exts = [".jpg", ".jpeg", ".png"]

all_images = []
for ext in image_exts:
    all_images.extend(list(IMAGES_DIR.rglob(f"*{ext}")))

all_labels = sorted(list(YOLO_ANN_DIR.rglob("*.txt")))

print("Toplam görüntü:", len(all_images))
print("Toplam YOLO label:", len(all_labels))

print("\nÖrnek görüntüler:")
for p in all_images[:10]:
    print(p)

print("\nÖrnek label dosyaları:")
for p in all_labels[:10]:
    print(p)

Toplam görüntü: 6265
Toplam YOLO label: 2245

Örnek görüntüler:
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/anonymous_002-007-1425-00_1733985413148_Frontal_View.jpg
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/anonymous_003-007-1003-00_1731041803319_Frontal_View.jpg
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/anonymous_003-007-1004-00_1731042060585_Frontal_View.jpg
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/anonymous_003-007-1006-00_1731044633484_Frontal_View.jpg
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/anonymous_003-007-1008-00_1731046769270_Frontal_View.jpg
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Images/no_retractors/Frontal/ano

In [5]:
label_class_counter = Counter()
invalid_lines = []
empty_label_files = 0
total_label_lines = 0

for label_path in all_labels:
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        empty_label_files += 1
        continue

    lines = text.splitlines()

    for line_idx, line in enumerate(lines):
        parts = line.strip().split()

        if len(parts) < 5:
            invalid_lines.append((label_path, line_idx, line))
            continue

        try:
            cls_id = int(float(parts[0]))
        except:
            invalid_lines.append((label_path, line_idx, line))
            continue

        label_class_counter[cls_id] += 1
        total_label_lines += 1

print("Toplam label satırı:", total_label_lines)
print("Boş label dosyası:", empty_label_files)
print("Geçersiz satır:", len(invalid_lines))

print("\nSınıf dağılımı:")
for cls_id, count in sorted(label_class_counter.items()):
    print(cls_id, ":", count)

Toplam label satırı: 6782
Boş label dosyası: 65
Geçersiz satır: 0

Sınıf dağılımı:
0 : 6228
1 : 554


In [6]:
shown = 0

for label_path in all_labels:
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        continue

    print("\nLABEL:", label_path)
    print("-" * 80)
    for line in text.splitlines()[:5]:
        print(line)

    shown += 1
    if shown >= 5:
        break


LABEL: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/no_retractors/Frontal/anonymous_003-007-1214-00_1732862917124_Frontal_View.txt
--------------------------------------------------------------------------------
0 0.2012863952219607 1.7460885571121791 0.28570608166312533 0.2991103384804173

LABEL: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/no_retractors/Frontal/anonymous_003-007-673-00_1729239709286_Frontal_View.txt
--------------------------------------------------------------------------------
1 0.9332763857251328 0.597972972972973 0.05789673500379651 0.24662162162162163

LABEL: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/no_retractors/Frontal/anonymous_003-007-848-00_1730185308013_Frontal_View.txt
--------------------------------------------------------------------------------
1 0.10112488521579426 0.5272

In [7]:
image_by_stem = {}

for img_path in all_images:
    image_by_stem[img_path.stem] = img_path

matched_pairs = []
unmatched_labels = []

for label_path in all_labels:
    stem = label_path.stem

    if stem in image_by_stem:
        matched_pairs.append((image_by_stem[stem], label_path))
    else:
        unmatched_labels.append(label_path)

print("Toplam görüntü:", len(all_images))
print("Toplam label:", len(all_labels))
print("Eşleşen image-label çifti:", len(matched_pairs))
print("Eşleşmeyen label:", len(unmatched_labels))

print("\nEşleşmeyen label örnekleri:")
for p in unmatched_labels[:20]:
    print(p)

Toplam görüntü: 6265
Toplam label: 2245
Eşleşen image-label çifti: 2227
Eşleşmeyen label: 18

Eşleşmeyen label örnekleri:
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/pilot/Mandibular/anonymous-1727845380707_Mandibular_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/pilot/Mandibular/anonymous-1727846130649_Mandibular_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/pilot/Mandibular/anonymous-1727849249910_Mandibular_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/pilot/Mandibular/anonymous-1727849691134_Mandibular_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted/intraoral_caries/Dataset/Annotations/Darknet_YOLO/pilot/Mandibular/anonymous-1727850241443_Mandibular_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_extracted

In [8]:
rows = []

for img_path, label_path in matched_pairs:
    rows.append({
        "image_path": str(img_path),
        "label_path": str(label_path),
        "stem": img_path.stem,
        "image_name": img_path.name,
        "source_folder": str(img_path.parent.relative_to(IMAGES_DIR))
    })

df_pairs = pd.DataFrame(rows)
df_pairs.head()

,image_path,label_path,stem,image_name,source_folder
0,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,anonymous_003-007-1214-00_1732862917124_Fronta...,anonymous_003-007-1214-00_1732862917124_Fronta...,no_retractors/Frontal
1,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,anonymous_003-007-673-00_1729239709286_Frontal...,anonymous_003-007-673-00_1729239709286_Frontal...,no_retractors/Frontal
2,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,anonymous_003-007-848-00_1730185308013_Frontal...,anonymous_003-007-848-00_1730185308013_Frontal...,no_retractors/Frontal
3,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,anonymous_003-007-870-00_1730264018655_Frontal...,anonymous_003-007-870-00_1730264018655_Frontal...,no_retractors/Frontal
4,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,/content/drive/MyDrive/DIS_PROJESI_007/data_ex...,anonymous_003-007-950-00_1730792603619_Frontal...,anonymous_003-007-950-00_1730792603619_Frontal...,no_retractors/Frontal


In [9]:
df_source_counts = df_pairs["source_folder"].value_counts().reset_index()
df_source_counts.columns = ["source_folder", "count"]
df_source_counts

,source_folder,count
0,no_retractors/Mandibular,495
1,retractors/Mandibular,495
2,retractors/Maxillary_Occlusal,378
3,no_retractors/Maxillary_Occlusal,372
4,pilot/Mandibular,128
5,pilot/Maxillary_Occlusal,88
6,retractors/Right_Lateral,48
7,no_retractors/Right_Lateral,46
8,no_retractors/Left_Lateral,44
9,retractors/Left_Lateral,40


In [10]:
CLASS_NAMES = {
    0: "caries"
}

print(CLASS_NAMES)

{0: 'caries'}


In [11]:
random.seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

n_total = len(indices)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.18)
n_test = n_total - n_train - n_val

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

df_pairs["split"] = ""

df_pairs.loc[train_idx, "split"] = "train"
df_pairs.loc[val_idx, "split"] = "val"
df_pairs.loc[test_idx, "split"] = "test"

df_pairs["split"].value_counts()

,count
split,
train,1558
val,400
test,269


In [12]:
if INTRAORAL_PREPARED.exists():
    shutil.rmtree(INTRAORAL_PREPARED)

for split in ["train", "val", "test"]:
    (INTRAORAL_PREPARED / "images" / split).mkdir(parents=True, exist_ok=True)
    (INTRAORAL_PREPARED / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Hazırlanan klasör:")
print(INTRAORAL_PREPARED)

Hazırlanan klasör:
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo


In [13]:
copy_summary = []

for split in ["train", "val", "test"]:
    df_split = df_pairs[df_pairs["split"] == split]

    copied_images = 0
    copied_labels = 0

    for _, row in df_split.iterrows():
        src_img = Path(row["image_path"])
        src_label = Path(row["label_path"])

        dst_img = INTRAORAL_PREPARED / "images" / split / src_img.name
        dst_label = INTRAORAL_PREPARED / "labels" / split / src_label.name

        shutil.copy2(src_img, dst_img)
        shutil.copy2(src_label, dst_label)

        copied_images += 1
        copied_labels += 1

    copy_summary.append({
        "split": split,
        "images": copied_images,
        "labels": copied_labels
    })

df_copy_summary = pd.DataFrame(copy_summary)
df_copy_summary

,split,images,labels
0,train,1558,1558
1,val,400,400
2,test,269,269


In [14]:
rows = []

for split in ["train", "val", "test"]:
    img_count = len(list((INTRAORAL_PREPARED / "images" / split).glob("*")))
    label_count = len(list((INTRAORAL_PREPARED / "labels" / split).glob("*.txt")))

    rows.append({
        "split": split,
        "images": img_count,
        "labels": label_count
    })

df_prepared_summary = pd.DataFrame(rows)
df_prepared_summary

,split,images,labels
0,train,1558,1558
1,val,400,400
2,test,269,269


In [15]:
prepared_class_counter = Counter()

for label_path in (INTRAORAL_PREPARED / "labels").rglob("*.txt"):
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        continue

    for line in text.splitlines():
        parts = line.strip().split()

        if len(parts) < 5:
            continue

        cls_id = int(float(parts[0]))
        prepared_class_counter[cls_id] += 1

rows = []

for cls_id, count in sorted(prepared_class_counter.items()):
    rows.append({
        "class_id": cls_id,
        "class_name": CLASS_NAMES.get(cls_id, f"class_{cls_id}"),
        "count": count
    })

df_prepared_class_counts = pd.DataFrame(rows)
df_prepared_class_counts

,class_id,class_name,count
0,0,caries,6174
1,1,class_1,554


In [16]:
names_text = "\n".join([f"  {cls_id}: {name}" for cls_id, name in CLASS_NAMES.items()])

data_yaml = f"""
path: {INTRAORAL_PREPARED}
train: images/train
val: images/val
test: images/test

names:
{names_text}
"""

data_yaml_path = INTRAORAL_PREPARED / "data.yaml"
data_yaml_path.write_text(data_yaml.strip())

print("data.yaml oluşturuldu:")
print(data_yaml_path)
print("\nİçerik:")
print(data_yaml_path.read_text())

data.yaml oluşturuldu:
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/data.yaml

İçerik:
path: /content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo
train: images/train
val: images/val
test: images/test

names:
  0: caries


In [17]:
def show_yolo_detection_sample(split="train"):
    img_dir = INTRAORAL_PREPARED / "images" / split
    label_dir = INTRAORAL_PREPARED / "labels" / split

    img_paths = sorted(list(img_dir.glob("*")))

    if len(img_paths) == 0:
        print("Görüntü yok:", split)
        return

    img_path = random.choice(img_paths)
    label_path = label_dir / (img_path.stem + ".txt")

    img = Image.open(img_path).convert("RGB")
    img_width, img_height = img.size

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    if label_path.exists():
        text = label_path.read_text(errors="ignore").strip()

        if text != "":
            for line in text.splitlines():
                parts = line.strip().split()

                cls_id = int(float(parts[0]))
                values = list(map(float, parts[1:]))

                # Detection formatı: class x_center y_center width height
                if len(values) == 4:
                    x_center, y_center, w_norm, h_norm = values

                    box_w = w_norm * img_width
                    box_h = h_norm * img_height
                    x_min = x_center * img_width - box_w / 2
                    y_min = y_center * img_height - box_h / 2

                    rect = patches.Rectangle(
                        (x_min, y_min),
                        box_w,
                        box_h,
                        linewidth=2,
                        edgecolor="red",
                        facecolor="none"
                    )
                    ax.add_patch(rect)

                    ax.text(
                        x_min,
                        y_min,
                        CLASS_NAMES.get(cls_id, str(cls_id)),
                        color="yellow",
                        fontsize=10,
                        bbox=dict(facecolor="red", alpha=0.5)
                    )

                # Eğer segmentation formatı çıkarsa polygon çizelim
                elif len(values) > 4:
                    coords = values
                    xs = coords[0::2]
                    ys = coords[1::2]

                    xs = [x * img_width for x in xs]
                    ys = [y * img_height for y in ys]

                    if len(xs) >= 3:
                        ax.plot(xs + [xs[0]], ys + [ys[0]], linewidth=2)

                        ax.text(
                            xs[0],
                            ys[0],
                            CLASS_NAMES.get(cls_id, str(cls_id)),
                            color="yellow",
                            fontsize=10,
                            bbox=dict(facecolor="red", alpha=0.5)
                        )

    ax.set_title(f"{split} | {img_path.name}")
    ax.axis("off")
    plt.show()

for split in ["train", "val", "test"]:
    for _ in range(2):
        show_yolo_detection_sample(split)

Output hidden; open in https://colab.research.google.com to view.

In [18]:
df_pairs.to_csv(RESULTS_DIR / "intraoral_image_label_pairs.csv", index=False)
df_source_counts.to_csv(RESULTS_DIR / "intraoral_source_folder_counts.csv", index=False)
df_prepared_summary.to_csv(RESULTS_DIR / "intraoral_prepared_summary.csv", index=False)
df_prepared_class_counts.to_csv(RESULTS_DIR / "intraoral_class_counts.csv", index=False)

print("Kaydedildi:")
print(RESULTS_DIR)

Kaydedildi:
/content/drive/MyDrive/DIS_PROJESI_007/results/intraoral_caries_prepare


In [19]:
class1_files = []

for label_path in (INTRAORAL_PREPARED / "labels").rglob("*.txt"):
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        continue

    has_class1 = False

    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        cls_id = int(float(parts[0]))
        if cls_id == 1:
            has_class1 = True
            break

    if has_class1:
        class1_files.append(label_path)

print("Class 1 içeren label dosyası sayısı:", len(class1_files))

for p in class1_files[:20]:
    print(p)

Class 1 içeren label dosyası sayısı: 326
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003-007-673-00_1729239709286_Frontal_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003-008-816-00_1729939059509_Frontal_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003-007-1363-00_1733726568839_Left_Lateral_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003-007-663-00_1729231959410_Left_Lateral_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003-007-987-00_1730958766991_Left_Lateral_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo/labels/train/anonymous_003_007_525_00_1728797698567_Left_Lateral_View.txt
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/intraoral_caries_yolo

In [20]:
def show_specific_label_image(label_path):
    split = label_path.parent.name
    img_dir = INTRAORAL_PREPARED / "images" / split

    # Aynı stem ile görüntü bul
    candidates = list(img_dir.glob(label_path.stem + ".*"))

    if len(candidates) == 0:
        print("Görüntü bulunamadı:", label_path.stem)
        return

    img_path = candidates[0]

    img = Image.open(img_path).convert("RGB")
    img_width, img_height = img.size

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    text = label_path.read_text(errors="ignore").strip()

    for line in text.splitlines():
        parts = line.strip().split()
        cls_id = int(float(parts[0]))
        values = list(map(float, parts[1:]))

        if len(values) == 4:
            x_center, y_center, w_norm, h_norm = values

            box_w = w_norm * img_width
            box_h = h_norm * img_height
            x_min = x_center * img_width - box_w / 2
            y_min = y_center * img_height - box_h / 2

            rect = patches.Rectangle(
                (x_min, y_min),
                box_w,
                box_h,
                linewidth=2,
                edgecolor="red",
                facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(
                x_min,
                y_min,
                f"class_{cls_id}",
                color="yellow",
                fontsize=10,
                bbox=dict(facecolor="red", alpha=0.5)
            )

    ax.set_title(f"{split} | {img_path.name}")
    ax.axis("off")
    plt.show()

for label_path in class1_files[:5]:
    show_specific_label_image(label_path)

Output hidden; open in https://colab.research.google.com to view.

In [21]:
for label_path in (INTRAORAL_PREPARED / "labels").rglob("*.txt"):
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        continue

    new_lines = []

    for line in text.splitlines():
        parts = line.strip().split()

        if len(parts) < 5:
            continue

        cls_id = int(float(parts[0]))

        # class 0 ve class 1'i caries olarak class 0'a çevir
        if cls_id in [0, 1]:
            parts[0] = "0"
            new_lines.append(" ".join(parts))

    label_path.write_text("\n".join(new_lines))

print("Class 1 etiketleri class 0 caries olarak birleştirildi.")

Class 1 etiketleri class 0 caries olarak birleştirildi.


In [22]:
prepared_class_counter = Counter()

for label_path in (INTRAORAL_PREPARED / "labels").rglob("*.txt"):
    text = label_path.read_text(errors="ignore").strip()

    if text == "":
        continue

    for line in text.splitlines():
        parts = line.strip().split()

        if len(parts) < 5:
            continue

        cls_id = int(float(parts[0]))
        prepared_class_counter[cls_id] += 1

df_prepared_class_counts = pd.DataFrame([
    {
        "class_id": cls_id,
        "class_name": "caries" if cls_id == 0 else f"class_{cls_id}",
        "count": count
    }
    for cls_id, count in sorted(prepared_class_counter.items())
])

df_prepared_class_counts

,class_id,class_name,count
0,0,caries,6728
